# Young customers: did the competitor stop new ones coming, or drive existing ones away?

**Context.** A competitor opened across the street from Lululime on **15 March 2024**. That notebook found that revenue from **18–24 year-olds fell ~45% and never recovered**, and that the loss came from **fewer active young customers**, not smaller baskets. This notebook asks *why* there are fewer of them.

**Questions this notebook answers**
1. **Winning new customers:** did fewer *new* 18–24 customers start shopping after the competitor opened?
2. **Keeping existing customers:** did *existing* 18–24 customers stop coming back? Were they already less loyal than other ages before the competitor?
3. **Is it still happening?** Is the number of active young customers still falling in 2026?

### Summary of the answers
- **New young customers halved and never recovered.** New 18–24 customers (Mar–Oct) fell from **420 in 2023 to 225 in 2024 (−46%)**, then 244 and 201. Their share of all new customers fell from 26% to 15%, while new customers aged 25+ grew.
- **Existing young customers became less loyal.** Before the competitor, 18–24s were **as likely as anyone to come back** (56% vs 54%). Across the opening their return rate fell to **35% vs 44%** for older customers, and it has stayed about 9 points behind.
- **It's still happening.** Active 18–24 customers went **1,513 → 1,106 → 858 → 725** (Mar–Oct 2023 → 2026), down 52%. With few new young customers joining, there are fewer to keep each year.

## Data and definitions

**The data.** Every Lululime transaction from **1 Nov 2022 to 31 Oct 2026**, plus one row per customer (age band, sign-up date, how they found the store). The team cleaned it once in shared code (`src/`):
- `load("lines")` returns one row per item line; `load("customers")` returns one row per customer.
- `C` holds agreed values, such as `C.PURCHASE` = "Purchase" and `C.DATA_START` / `C.DATA_END` = the first and last day of sales data. The competitor's opening day, 15 March 2024, is `C.COMPETITOR_OPEN_DAY` (`OPEN_DAY` below).

**Purchases only.** "Bought" always means a purchase (not a return or a free repair visit).

**Terms**
| Term | Meaning |
|---|---|
| New customer | A customer's `signup_date` is the date of their **first purchase**. A customer is "new" in the period that date falls in |
| Why only sign-ups from Nov 2022 | The customer file only includes people who bought something between Nov 2022 and Oct 2026. Anyone who signed up in, say, 2020 and stopped shopping before 2022 is missing. Counting older sign-ups would undercount them (survivorship bias) |
| Active customer | Made at least one purchase in the period |
| Returning customer | Active in the period, but signed up before it |
| Retention | Of the customers who bought in a window, the share who **bought again in the same window one year later**. Using the same months each year cancels out seasonality |
| Nov–Feb windows | Used for retention because **Nov 2022 – Feb 2023 → Nov 2023 – Feb 2024** is the only year-on-year pair that sits entirely **before** the competitor opened. It gives a "normal" benchmark |
| Mar–Oct windows | Used to match `a1_competitor_impact.ipynb`: the longest window available in every year 2023–2026 (the data ends in October 2026) |

In [ ]:
import sys
from pathlib import Path

# Make the repo's shared code (src/) importable; this notebook lives in analysis/a_competitor/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.metrics import load      # shared loader for the cleaned tables
from src import config as C       # shared constants: dates, row types, paths

OUT = C.RESULTS_DIR / "a_competitor"     # charts are saved here
OUT.mkdir(parents=True, exist_ok=True)
OPEN_DAY = C.COMPETITOR_OPEN_DAY        # the competitor's actual opening day, 15 March 2024 (C.COMPETITOR_OPEN = the month)
AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]
AGE_COLORS = {"18-24": "#D1495B", "25-34": "#E09F3E", "35-44": "#2B4C7E", "45-54": "#5B8DB8", "55+": "#9AA5B1"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False})

lines = load("lines")
customers = load("customers")
age = customers.set_index("customer_id").age_band      # lookup: customer -> age band
p = lines[lines.row_type == C.PURCHASE]                # purchases only

## 1. New customers per quarter, by age band
Each customer is counted once, in the quarter of their first purchase.

In [ ]:
new = customers[customers.signup_date >= C.DATA_START].copy()     # sign-ups from Nov 2022 only (see definitions)
new["quarter"] = new.signup_date.dt.to_period("Q")
q = new.pivot_table(index="quarter", columns="age_band", values="customer_id", aggfunc="count").fillna(0)
q = q.loc[: (C.DATA_END - pd.DateOffset(months=1)).to_period("Q") - 1, AGE_ORDER]   # drop the partial last quarter (Oct 2026 only)

fig, ax = plt.subplots(figsize=(10, 4.3))
x = q.index.to_timestamp()
for a in AGE_ORDER:
    hit = a == "18-24"
    ax.plot(x, q[a], marker="o", ms=3.5, color=AGE_COLORS[a], lw=2.6 if hit else 1.3, label=a, zorder=3 if hit else 2)
ax.axvline(OPEN_DAY, color="#555", ls="--", lw=0.9)
ax.text(OPEN_DAY, ax.get_ylim()[1] * 0.97, " Competitor opens\n (15 Mar 2024)", fontsize=8.5, va="top")
ax.set(title="New 18–24 customers roughly halved after the competitor opened, and never recovered",
       ylabel="New customers per quarter")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.22), fontsize=9)
plt.tight_layout(); fig.savefig(OUT / "a3_new_customers_by_age.png", bbox_inches="tight"); plt.show()

**What this shows.** Before March 2024, Lululime won 124–178 new 18–24 customers a quarter, roughly as many as 35–44s. After the competitor opened, that fell to 52–118 a quarter. New customers in the other age bands kept their usual level, or grew.

In [ ]:
# Same comparison as a1_competitor_impact.ipynb: Mar–Oct of each year
mo = new[new.signup_date.dt.month.between(3, 10)]
by_year = mo.pivot_table(index=mo.signup_date.dt.year, columns="age_band", values="customer_id", aggfunc="count")[AGE_ORDER]
by_year.index.name = "Mar–Oct of"
share = by_year.div(by_year.sum(axis=1), axis=0)
vs23 = by_year / by_year.loc[2023] - 1

display(pd.concat({"New customers": by_year, "Share of new customers": share, "vs. 2023": vs23}, axis=1)
        .style.format("{:,.0f}", subset="New customers")
        .format("{:.0%}", subset="Share of new customers")
        .format("{:+.0%}", subset="vs. 2023")
        .set_caption("<b>New customers in Mar–Oct, by age band</b>")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))

# How did those new 18-24 customers find the store?
ch = mo[mo.age_band == "18-24"].pivot_table(index=mo.signup_date.dt.year, columns="acquisition_channel",
                                             values="customer_id", aggfunc="count").fillna(0).astype(int)
ch.index.name = "Mar–Oct of"
print("New 18-24 customers by how they found the store (acquisition channel):")
ch

**What this shows.**
- **New 18–24 customers:** 420 (2023) → 225 (2024), −46%. They stayed around 200–245 in 2025 and 2026.
- **Share of all new customers:** 18–24s fell from 26% to ~15%.
- **How they found the store:** walk-ins (−50%), social media (−45%) and referrals (−38%) all fell sharply in 2024. The competitor reduced young shoppers' interest across the board, not through one channel.

## 2. Do existing customers come back?
Of the customers who bought in a window, the share who bought again in the same window a year later, by age band. "25+ (all other ages)" combines everyone aged 25 and over.

In [ ]:
def active(start, end):
    # customers with at least one purchase between start (inclusive) and end (exclusive)
    return set(p[(p.date >= start) & (p.date < end)].customer_id)

def retention(w_from, w_to):
    base = pd.Series(sorted(active(*w_from)))          # customers who bought in the first window
    back = base.isin(active(*w_to))                    # ...and bought again in the second
    r = back.groupby(base.map(age).values).mean().reindex(AGE_ORDER)
    r["25+ (all other ages)"] = back[base.map(age).isin(AGE_ORDER[1:]).values].mean()
    return r

NOV_FEB = {2022: ("2022-11-01", "2023-03-01"), 2023: ("2023-11-01", "2024-03-01"),
           2024: ("2024-11-01", "2025-03-01"), 2025: ("2025-11-01", "2026-03-01")}
ret_nf = pd.DataFrame({
    "Nov 22–Feb 23 → Nov 23–Feb 24\n(before competitor)": retention(NOV_FEB[2022], NOV_FEB[2023]),
    "Nov 23–Feb 24 → Nov 24–Feb 25\n(competitor opens in between)": retention(NOV_FEB[2023], NOV_FEB[2024]),
    "Nov 24–Feb 25 → Nov 25–Feb 26\n(after)": retention(NOV_FEB[2024], NOV_FEB[2025]),
})
MAR_OCT = {y: (f"{y}-03-01", f"{y}-11-01") for y in (2023, 2024, 2025, 2026)}
ret_mo = pd.DataFrame({f"Mar–Oct {y} → {y + 1}": retention(MAR_OCT[y], MAR_OCT[y + 1]) for y in (2023, 2024, 2025)})

display(ret_nf.style.format("{:.0%}").set_caption("<b>Retention, Nov–Feb windows</b> (includes a pre-competitor benchmark)")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))
display(ret_mo.style.format("{:.0%}").set_caption("<b>Retention, Mar–Oct windows</b> (same window as a1_competitor_impact.ipynb)")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))
gap = ret_nf.loc["18-24"] - ret_nf.loc["25+ (all other ages)"]
print("18-24 minus 25+ retention gap (Nov–Feb):", ", ".join(f"{g:+.0%}" for g in gap))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4))
labels = ["Before competitor\n(Nov 22–Feb 23 → next winter)", "Competitor opens in between\n(Nov 23–Feb 24 → next winter)",
          "After\n(Nov 24–Feb 25 → next winter)"]
xx = np.arange(3); w = 0.36
for i, (row, col, name) in enumerate([("18-24", "#D1495B", "18–24"), ("25+ (all other ages)", "#2B4C7E", "25 and over")]):
    vals = ret_nf.loc[row].values
    ax.bar(xx + (i - 0.5) * w, vals, w, color=col, label=name)
    for xi, v in zip(xx + (i - 0.5) * w, vals):
        ax.text(xi, v + 0.01, f"{v:.0%}", ha="center", fontsize=9)
ax.set_xticks(xx, labels, fontsize=8.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylim=(0, 0.68), ylabel="Bought again next winter",
       title="Young customers were as loyal as everyone before the competitor; afterwards they were the least likely to return")
ax.legend(loc="upper right")
plt.tight_layout(); fig.savefig(OUT / "a3_retention_by_age.png", bbox_inches="tight"); plt.show()

**What this shows.**
- **Before the competitor:** 56% of 18–24s came back the next winter, the same as older customers (54%). Young customers weren't naturally less loyal.
- **Across the opening:** every age band's return rate dropped, but 18–24s dropped **twice as much** (−21 points vs −10), to 35% vs 44%.
- **After:** both groups improved, but 18–24s stayed ~9 points behind (44% vs 52%). The Mar–Oct windows show the same gap.

**Caveat:** the pre-competitor benchmark is a single year-on-year comparison, because the data starts in November 2022.

## 3. Where did the active 18–24 customers go?
Active 18–24 customers in each Mar–Oct = **new** that year (first purchase in that window) + **returning** (signed up earlier).

In [ ]:
rows = {}
for y in (2023, 2024, 2025, 2026):
    ids = pd.Series(sorted(active(*MAR_OCT[y])))
    ids = ids[ids.map(age).eq("18-24").values]                 # active 18-24 customers that year
    sd = customers.set_index("customer_id").signup_date.reindex(ids)
    is_new = (sd >= MAR_OCT[y][0]).values                     # first purchase inside this window = new
    rows[y] = {"new": int(is_new.sum()), "returning": int((~is_new).sum())}
split = pd.DataFrame(rows).T
split["total active"] = split.sum(axis=1)
display(split)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(split.index.astype(str), split["returning"], color="#5B8DB8", label="Returning (signed up earlier)")
ax.bar(split.index.astype(str), split["new"], bottom=split["returning"], color="#D1495B", label="New that year")
for i, (y, r) in enumerate(split.iterrows()):
    ax.text(i, r["total active"] + 15, f"{r['total active']:,}", ha="center", fontsize=9, fontweight="bold")
    ax.text(i, r["returning"] + r["new"] / 2, f"{r['new']}", ha="center", va="center", color="white", fontsize=8.5)
    ax.text(i, r["returning"] / 2, f"{r['returning']}", ha="center", va="center", color="white", fontsize=8.5)
ax.set(title="Active 18–24 customers (Mar–Oct) shrank as fewer new ones replaced those who left",
       ylabel="Active 18–24 customers", ylim=(0, split["total active"].max() * 1.15))
ax.legend(loc="upper right", fontsize=8.5)
plt.tight_layout(); fig.savefig(OUT / "a3_18_24_new_vs_returning.png", bbox_inches="tight"); plt.show()

d = split.loc[2024] - split.loc[2023]
print(f"2023 -> 2024: active 18-24 {d['total active']:+,}  (new {d['new']:+,}, returning {d['returning']:+,})")

**What this shows.**
- **2023 → 2024:** active 18–24 customers fell by 407. The loss split almost evenly: 195 fewer new customers and 212 fewer returning ones.
- **After 2024:** the decline continued, to 725 in 2026 (−52% vs 2023). Each year fewer new young customers join, so the pool of returning ones keeps shrinking too.
- **The free repair program (Apr 2025) didn't reverse this:** active 18–24s kept falling in 2025 and 2026.

## 4. Key numbers

In [ ]:
summary = {
    "New 18-24 customers, Mar–Oct 2023 → 2024 / 2025 / 2026":
        f"{by_year.loc[2023, '18-24']} → {by_year.loc[2024, '18-24']} / {by_year.loc[2025, '18-24']} / {by_year.loc[2026, '18-24']} "
        f"({vs23.loc[2024, '18-24']:+.0%} / {vs23.loc[2025, '18-24']:+.0%} / {vs23.loc[2026, '18-24']:+.0%})",
    "18-24 share of new customers, 2023 → 2026": f"{share.loc[2023, '18-24']:.0%} → {share.loc[2026, '18-24']:.0%}",
    "New 25+ customers, Mar–Oct 2024 vs 2023": f"{by_year.loc[2024, AGE_ORDER[1:]].sum() / by_year.loc[2023, AGE_ORDER[1:]].sum() - 1:+.0%}",
    "Retention 18-24 vs 25+, before competitor": f"{ret_nf.iloc[:, 0]['18-24']:.0%} vs {ret_nf.iloc[:, 0]['25+ (all other ages)']:.0%}",
    "Retention 18-24 vs 25+, across competitor opening": f"{ret_nf.iloc[:, 1]['18-24']:.0%} vs {ret_nf.iloc[:, 1]['25+ (all other ages)']:.0%}",
    "Retention 18-24 vs 25+, after": f"{ret_nf.iloc[:, 2]['18-24']:.0%} vs {ret_nf.iloc[:, 2]['25+ (all other ages)']:.0%}",
    "Active 18-24 change 2023→2024 (new / returning)": f"{d['new']:+,} / {d['returning']:+,}",
    "Active 18-24, Mar–Oct 2023 → 2026": f"{split.loc[2023, 'total active']:,} → {split.loc[2026, 'total active']:,}",
}
pd.Series(summary, name="value").to_frame()

## Conclusions and caveats
1. The competitor damaged Lululime's young customer base in **two ways**: it **halved new 18–24 customers**, and it made existing ones **less likely to return** (they had been as loyal as everyone else).
2. Neither effect has recovered, so the young segment **keeps shrinking** (−52% active customers in three years).
3. **What this means for leadership's plan:** expanding loyalty perks rewards customers who already shop at Lululime. It doesn't address winning new young customers, which is where the lasting damage is. A metric worth tracking: **new 18–24 customers per quarter**.

**Caveats.** Age band is the age at first purchase and doesn't update. New-customer counts start in November 2022. The pre-competitor retention benchmark is a single comparison. These patterns line up with the competitor's opening in time, but we have no data on the competitor itself.

---
*Team reference: Stream A should-have (sign-ups and visits by age band). Findings F9–F10 in `results/a_competitor/findings.md`. Charts saved to `results/a_competitor/`.*